Pull likely nucleoside/nucleotide analog antivirals from ChEMBL and save to CSV.

In [3]:
# Install ChEMBL Client if needed 
#!pip install chembl_webresource_client pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 1.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 1.3 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.12.2
    Uninstalling typing_extensions-4.12.2:
      Successfully uninstalled typing_extensions-4.12.2
  Attempting uninstall: attrs
    Found existing installation: attrs 23.2.0
    Uninstalling attrs-23.2.0:
      Successfully uninstalled attrs-23.2.0


In [4]:
# Import needed packages
import pandas as pd
from chembl_webresource_client.new_client import new_client

In [5]:
# Connect to API
molecule = new_client.molecule
mechanism = new_client.mechanism

In [6]:
# Pull all approved Anti-virals based on the ATC Class (ATC class J05 = antivirals)
antivirals = molecule.filter(
    atc_classifications__level5__startswith="J05",
    max_phase=4
)

print("Example record:")
print(next(iter(antivirals)))

Example record:
{'atc_classifications': ['J05AG01'], 'availability_type': 1, 'biotherapeutic': None, 'black_box_warning': 1, 'chemical_probe': 0, 'chirality': 2, 'cross_references': [{'xref_id': 'nevirapine', 'xref_name': 'nevirapine', 'xref_src': 'DailyMed'}, {'xref_id': 'human/EPAR/nevirapine-teva', 'xref_name': 'nevirapine-teva', 'xref_src': 'EMA'}, {'xref_id': 'human/EPAR/viramune', 'xref_name': 'viramune', 'xref_src': 'EMA'}], 'dosed_ingredient': True, 'first_approval': 1996, 'first_in_class': 0, 'helm_notation': None, 'inorganic_flag': 0, 'max_phase': '4.0', 'molecule_chembl_id': 'CHEMBL57', 'molecule_hierarchy': {'active_chembl_id': 'CHEMBL57', 'molecule_chembl_id': 'CHEMBL57', 'parent_chembl_id': 'CHEMBL57'}, 'molecule_properties': {'alogp': '2.65', 'aromatic_rings': 2, 'full_molformula': 'C15H14N4O', 'full_mwt': '266.30', 'hba': 4, 'hbd': 1, 'heavy_atoms': 20, 'mw_freebase': '266.30', 'np_likeness_score': '-0.25', 'num_ro5_violations': 0, 'psa': '58.12', 'qed_weighted': '0.86'

In [7]:
# Define keywords for nucleoside/nucleotide analogs to narrow down the molecules we are training on
keywords = [
    "nucleoside",
    "nucleotide",
    "polymerase",
    "reverse transcriptase",
    "chain terminator"
]

seed_drugs = [
    "remdesivir",
    "ribavirin",
    "favipiravir",
    "molnupiravir",
    "sofosbuvir",
    "cidofovir",
    "tenofovir",
    "adefovir",
    "lamivudine",
    "emtricitabine",
]

In [8]:
# Filter all antivirals based on Keywords
records = []

for mol in antivirals:

    name = mol.get("pref_name")
    structures = mol.get("molecule_structures")

    if not name or not structures:
        continue

    smiles = structures.get("canonical_smiles")

    text = name.lower()

    # keep if name matches seed list or contains keyword
    if any(seed in text for seed in seed_drugs) or any(k in text for k in keywords):

        records.append({
            "chembl_id": mol.get("molecule_chembl_id"),
            "name": name,
            "smiles": smiles
        })

df = pd.DataFrame(records)

print("Molecules found:", len(df))
df.head()

Molecules found: 11


,chembl_id,name,smiles
0,CHEMBL141,LAMIVUDINE,Nc1ccn([C@@H]2CS[C@H](CO)O2)c(=O)n1
1,CHEMBL885,EMTRICITABINE,Nc1nc(=O)n([C@@H]2CS[C@H](CO)O2)cc1F
2,CHEMBL922,ADEFOVIR DIPIVOXIL,CC(C)(C)C(=O)OCOP(=O)(COCCn1cnc2c(N)ncnc21)OCO...
3,CHEMBL203321,BRINCIDOFOVIR,CCCCCCCCCCCCCCCCOCCCOP(=O)(O)CO[C@H](CO)Cn1ccc...
4,CHEMBL221722,FAVIPIRAVIR,NC(=O)c1nc(F)cnc1O
